In [0]:
import os
import sys

sys.path.insert(0, os.path.abspath("../app"))

from datetime import datetime, timezone

import numpy as np
import pandas as pd

from squawk_lib import config
from squawk_lib.config import DETECT
from squawk_lib.detect import add_features, detect_window, haversine_nm

spark.conf.set("spark.sql.session.timeZone", "UTC")

LOOKBACK_HOURS = 12
now = datetime.now(timezone.utc).replace(tzinfo=None)
window_start = pd.Timestamp(now) - pd.Timedelta(hours=LOOKBACK_HOURS)

positions = spark.sql(f"""
    SELECT icao24, callsign, event_ts, lat, lon, alt_ft, vr_fpm, track_deg, on_ground
    FROM {config.TABLES['positions']}
    WHERE event_ts > TIMESTAMP'{window_start:%Y-%m-%d %H:%M:%S}'
""").toPandas()

print(f"{len(positions):,} positions from {positions['icao24'].nunique():,} aircraft, "
      f"{window_start:%H:%M}Z to {now:%H:%M}Z")

In [0]:
from squawk_lib.detect import add_features
from squawk_lib.config import DETECT

probe = []
for icao, g in positions.groupby("icao24"):
    if len(g) < 5:
        continue
    f = add_features(g)
    in_zone = ((f["dist_stack_nm"] <= DETECT["stack_radius_nm"])
               & (f["alt_ft"] >= DETECT["stack_min_alt_ft"])
               & (~f["on_ground"]))
    if in_zone.any():
        probe.append({
            "icao24": icao,
            "callsign": f["callsign"].dropna().iloc[0] if f["callsign"].notna().any() else None,
            "stack": f.loc[in_zone, "nearest_stack"].mode().iloc[0],
            "points_in_zone": int(in_zone.sum()),
            "min_dist_nm": round(float(f["dist_stack_nm"].min()), 1),
            "max_turn_deg": round(float(f.loc[in_zone, "cum_turn_deg"].abs().max()), 0),
        })

probe = pd.DataFrame(probe).sort_values("max_turn_deg", ascending=False)
print(len(probe), "aircraft entered a stack zone in the last 3 hours")
display(probe.head(20))